# 06 · Simulación de producción

Documenta la simulación de producción (sección 5) y la evidencia del reentrenamiento (sección 7) y de la trazabilidad (sección 10.6).

La simulación la ejecuta Kestra, no este notebook:

| Paso | Archivo | Qué hace |
|---|---|---|
| Construir los lotes (una vez) | `src/lotes.py` | Divide el período de prueba en 8 lotes e induce la deriva; escribe `OPERACION.LOTES_SIMULACION` |
| Paso N = 1…9 | `kestra/produccion_lotes.yml` | 1) llegan las ventas reales del lote N-1 → 2) monitoreo de su desempeño y disparador → 3) reentrenamiento si corresponde (con pausa de aprobación) → 4) pronóstico del lote N con la API → 5) monitoreo del lote N |
| Monitoreo | `kestra/monitoreo.yml`, `monitoreo/` | Métricas, deriva y alertas en `OPERACION.MONITOREO` y `OPERACION.ALERTAS` |
| Reentrenamiento | `kestra/reentrenamiento.yml`, `src/reentrenar.py` | Challenger, comparación, pausa y decisión |

Las ventas reales de un lote llegan en el paso siguiente: así se respeta el horizonte y el reentrenamiento solo usa lotes
cuyo resultado ya se conoce.

In [1]:
import sys
sys.path.append("..")

import warnings
warnings.filterwarnings("ignore")

import mlflow
import pandas as pd
from mlflow import MlflowClient

from src.datos import conectar
from src.lotes import (DIAS_POR_LOTE, FACTOR_DISTANCIA, FACTOR_VENTAS, LOTES_CON_DERIVA, TIPO_AFECTADO,
                       calendario_lotes)
from src.pipeline import configurar_mlflow

pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", 120)


def consultar(sql: str, esquema: str = "OPERACION") -> pd.DataFrame:
    with conectar(esquema) as conn:
        df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = df.columns.str.lower()
    return df

## 1. Construcción de los lotes

El modelo no recibe tráfico real, así que se simula con el período de prueba, que no se usó para entrenar ni para elegir
hiperparámetros: del 2015-06-20 al 2015-07-31 (42 días). Seis semanas no alcanzan para los 8 lotes mínimos, por eso se usan
bloques de 5 o 6 días: una simulación comprimida del ciclo semanal. Los datos son reales; solo se alteran los lotes con deriva.

In [2]:
lotes = consultar("""
    SELECT LOTE, MIN(SALES_DATE) AS DESDE, MAX(SALES_DATE) AS HASTA, COUNT(*) AS TIENDA_DIA,
           COUNT(DISTINCT STORE_ID) AS TIENDAS, SUM(IFF(ALTERADO, 1, 0)) AS FILAS_ALTERADAS,
           AVG(IS_PROMO::INT) AS PCT_DIAS_CON_PROMO, SUM(SALES) AS VENTAS
    FROM LOTES_SIMULACION GROUP BY LOTE ORDER BY LOTE
""").set_index("lote")
lotes["dias"] = DIAS_POR_LOTE
lotes

,desde,hasta,tienda_dia,tiendas,filas_alteradas,pct_dias_con_promo,ventas,dias
lote,,,,,,,,
1,2015-06-20,2015-06-24,4488,1114,0,0.000000,2.5743e+07,5
2,2015-06-25,2015-06-29,4491,1115,0,0.248274,3.1564e+07,5
3,2015-06-30,2015-07-04,5572,1115,0,0.800072,4.3505e+07,5
4,2015-07-05,2015-07-09,4489,1114,0,0.000000,2.8057e+07,5
5,2015-07-10,2015-07-14,4485,1114,0,0.496321,3.3786e+07,5
6,2015-07-15,2015-07-19,4482,1114,2410,0.744534,2.8488e+07,5
7,2015-07-20,2015-07-25,6678,1113,3600,0.000000,3.4958e+07,6
8,2015-07-26,2015-07-31,5597,1113,3012,0.994283,4.2485e+07,6


## 2. Deriva inducida (escenario de prueba, no un hecho de Rossmann)

En los lotes **6, 7 y 8** abre un competidor cerca de las tiendas tipo **"a"** (602 tiendas, el 54 % de los datos):

| Qué se alteró | Cómo | Tipo de deriva | Qué debe detectar el monitoreo |
|---|---|---|---|
| `competition_distance_m` (entrada) | × 0,1 | Deriva de datos | PSI de la distancia > 0,25 y valores fuera del rango de entrenamiento |
| `sales` (ventas reales) | × 0,8 | Cambio en la relación entre las entradas y el target | WAPE > 0,0945 (media de la CV + 2 desviaciones) |

Las features de ventas con rezago de 42 días no cambian: usan días anteriores al competidor. Las tablas de origen (OBT) no se
modifican; la alteración se hace sobre la copia de `LOTES_SIMULACION` y cada fila alterada queda marcada con `ALTERADO = TRUE`.

In [3]:
# Antes y despues, en las tiendas afectadas y en el resto
consultar("""
    SELECT LOTE, STORE_TYPE = 'a' AS TIENDA_TIPO_A,
           MEDIAN(COMPETITION_DISTANCE_M) AS DISTANCIA_MEDIANA_M,
           AVG(SALES) AS VENTA_MEDIA, AVG(SALES_MEAN_LAG42_DOW) AS VENTA_REGLA_MEDIA,
           SUM(IFF(ALTERADO, 1, 0)) AS FILAS_ALTERADAS
    FROM LOTES_SIMULACION GROUP BY 1, 2 ORDER BY 2 DESC, 1
""")

,lote,tienda_tipo_a,distancia_mediana_m,venta_media,venta_regla_media,filas_alteradas
0,1,True,1790.0,5644.3522,7024.4792,0
1,2,True,1790.0,6911.7896,7328.7981,0
2,3,True,1790.0,7740.7399,7457.7365,0
3,4,True,1800.0,6286.5159,7539.5296,0
4,5,True,1800.0,7336.1554,7357.7917,0
5,6,True,179.5,5541.1778,6900.6916,2410
6,7,True,179.0,4620.2860,7201.2912,3600
7,8,True,179.0,6682.5288,7593.8084,3012
8,1,False,3290.0,5842.7070,7240.3213,0
9,2,False,3290.0,7164.3277,7731.2827,0


## 3. Ejecución lote por lote

In [4]:
ejecucion = consultar("""
    SELECT TRY_TO_NUMBER(p.LOTE) AS LOTE, COUNT(*) AS PREDICCIONES,
           LISTAGG(DISTINCT p.VERSION_MODELO, ',') AS VERSION_MODELO,
           SUM(IFF(p.RESPALDO, 1, 0)) AS EN_RESPALDO,
           MIN(p.TS) AS PRONOSTICADO,
           AVG(IFF(p.DECISION = 'reforzar_turnos', 1, 0)) AS REFORZAR,
           AVG(IFF(p.DECISION = 'mantener_plan', 1, 0)) AS MANTENER,
           AVG(IFF(p.DECISION = 'reducir_turnos', 1, 0)) AS REDUCIR
    FROM PREDICCIONES p WHERE TRY_TO_NUMBER(p.LOTE) IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").set_index("lote")
llegadas = consultar("SELECT LOTE, MIN(LLEGADA_TS) AS LLEGAN_VENTAS_REALES FROM VENTAS_REALES GROUP BY LOTE").set_index("lote")
ejecucion.join(llegadas)

,predicciones,version_modelo,en_respaldo,pronosticado,reforzar,mantener,reducir,llegan_ventas_reales
lote,,,,,,,,
1,4488,1,0,2026-11-24 03:22:58.870333,0.000446,0.261809,0.737745,2026-11-23 19:23:36.934
2,4491,1,0,2026-11-24 03:23:57.954359,0.188599,0.419060,0.392340,2026-11-23 19:24:30.421
3,5572,1,0,2026-11-24 03:24:52.599026,0.314070,0.684494,0.001436,2026-11-23 19:25:30.389
4,4489,1,0,2026-11-24 03:25:54.879202,0.001337,0.442415,0.556249,2026-11-23 19:26:27.809
5,4485,1,0,2026-11-24 03:56:06.976021,0.335340,0.479153,0.185507,2026-11-23 19:56:41.900
6,4482,1,0,2026-11-24 03:57:04.283647,0.124275,0.863677,0.012048,2026-11-23 19:57:41.615
7,6678,1,0,2026-11-24 03:58:06.187721,0.000000,0.255765,0.744235,2026-11-23 19:58:45.152
8,5597,4,0,2026-11-24 04:11:27.418873,0.386278,0.611220,0.002501,2026-11-23 20:12:10.021


Las ventas reales de cada lote llegan después de pronosticarlo (en el paso siguiente). El lote 8 se pronosticó con la v4,
el modelo aprobado en el reentrenamiento del paso 8.

## 4. Reentrenamientos

**Disparador** (sección 7.1): WAPE > 0,0945 durante 2 lotes seguidos, o por calendario cada 4 lotes con ventas reales.
La deriva de datos sin caída de desempeño no dispara el reentrenamiento.
**Ventana:** histórico completo del PSet 3 más los lotes etiquetados anteriores al de evaluación, con peso 10.
**Regla de promoción** (sección 4): el challenger reemplaza al champion si su costo de turnos en el lote de evaluación es
menor por más de 0,0039 (una desviación estándar del costo en la CV del champion). La decisión final es humana.

In [5]:
configurar_mlflow()
cliente = MlflowClient()
corridas = mlflow.search_runs(experiment_names=["rossmann"], filter_string="tags.fase = 'reentrenamiento'",
                              order_by=["start_time ASC"])
reentrenos = corridas[[
    "tags.mlflow.runName", "tags.motivo", "tags.champion_previo", "params.lotes_entrenamiento",
    "params.lote_evaluacion", "metrics.costo_champion_lote_eval", "metrics.costo_challenger_lote_eval",
    "metrics.mejora_costo", "metrics.umbral_promocion", "metrics.cumple_regla", "tags.decision", "tags.motivo_decision",
]]
reentrenos.columns = [c.split(".", 1)[-1] for c in reentrenos.columns]
reentrenos.T

,0,1,2
mlflow.runName,reentrenamiento_lote_4,reentrenamiento_lote_7,reentrenamiento_lote_8
motivo,Calendario: reentrenamiento programado cada 4 lotes (lote 4),Desempeno: WAPE 0.1740 y 0.1513 en los lotes 6 y 7 superan 0.0945 (media CV + 2 desv.),Desempeno: WAPE 0.1513 y 0.1553 en los lotes 7 y 8 superan 0.0945 (media CV + 2 desv.)
champion_previo,1,1,4
lotes_entrenamiento,"1,2,3","1,2,3,4,5,6","1,2,3,4,5,6,7"
lote_evaluacion,4,7,8
costo_champion_lote_eval,0.1277,0.1829,0.1773
costo_challenger_lote_eval,0.1258,0.1632,0.1751
mejora_costo,0.002,0.0198,0.0021
umbral_promocion,0.0039,0.0039,0.0039
cumple_regla,0.0,1.0,0.0


In [6]:
versiones = cliente.search_model_versions("name = 'rossmann-modelo'")
pd.DataFrame([{
    "version": int(v.version),
    "run_id": v.run_id,
    "aliases": ", ".join(cliente.get_model_version(v.name, v.version).aliases),
    "decision": v.tags.get("decision", ""),
    "motivo_decision": v.tags.get("motivo_decision", ""),
} for v in versiones]).sort_values("version").set_index("version")

,run_id,aliases,decision,motivo_decision
version,,,,
1,0e34d952530740ec900db92ec5831a15,,,
2,2e9af5153c03402aa4e0f653be8d90bf,,,
3,66adf1fee7fa45a281b13664a485d8d4,,rechazado,"Reentrenamiento por calendario. El challenger v3 reduce el costo de turnos solo 0,0020 en el lote 4, menos que el um..."
4,1139e41bd4314d359c9cf5951d123e05,champion,aprobado,Reentrenamiento por caída de desempeño (WAPE sobre el umbral en los lotes 6 y 7). El challenger v4 reduce el costo d...
5,92f711be508a44cc9b03df43e4af6485,challenger,rechazado,"El challenger v5 reduce el costo de turnos solo 0,0021 en el lote 8, menos que el umbral de 0,0039; la diferencia es..."


Resultado del ciclo sobre el lote con deriva (sección 7.4): **alerta** (WAPE de 0,174 y 0,151 en los lotes 6 y 7) →
**reentrenamiento** → **challenger v4** → **comparación** en el lote 7 (costo 0,1632 frente a 0,1829) → **aprobación** →
`champion` pasa a la v4 y la API la carga sin reconstruir la imagen. Los reentrenamientos de los pasos 5 (calendario) y 9
(desempeño) se rechazaron porque la mejora quedó por debajo del umbral.

**Rollback** (sección 4): se demostró moviendo el alias `champion` a la v2 y de vuelta a la v1 en el Model Registry, y llamando
a `POST /recargar`; `/modelo` mostró la versión 2 y luego la 1 sin reconstruir la imagen (capturas en el documento técnico).

## 5. Trazabilidad de una predicción (sección 10.6)

Una predicción de `PREDICCIONES` → versión del modelo y `run_id` → corrida en MLflow → snapshot de la OBT con el que se
entrenó → tablas Gold de las que se derivó la OBT.

In [7]:
pred = consultar("""
    SELECT ID_PREDICCION, TS, LOTE, STORE_ID, FECHA, VENTAS_PRONOSTICADAS, DECISION, VERSION_MODELO, RUN_ID
    FROM PREDICCIONES WHERE LOTE = '1' ORDER BY STORE_ID, FECHA LIMIT 1
""").iloc[0]
corrida = cliente.get_run(pred["run_id"])
origen = cliente.get_run(corrida.data.params["origen_run_id"])  # corrida de entrenamiento del champion (tuning)
obt = consultar("SELECT MAX(OBT_BUILT_AT) AS CONSTRUIDA, ANY_VALUE(OBT_RUN_ID) AS OBT_RUN_ID, COUNT(*) AS FILAS "
                "FROM OBT_SALES_DAILY", esquema="OBT").iloc[0]

pd.Series({
    "1. id_prediccion": pred["id_prediccion"],
    "   tienda / fecha / lote": f"{pred['store_id']} / {pred['fecha']} / {pred['lote']}",
    "   pronostico / decision": f"{pred['ventas_pronosticadas']:.0f} / {pred['decision']}",
    "2. version del modelo": pred["version_modelo"],
    "   run_id": pred["run_id"],
    "3. corrida en MLflow": corrida.data.tags["mlflow.runName"],
    "   corrida de entrenamiento": f"{origen.data.tags['mlflow.runName']} ({origen.info.run_id})",
    "4. tabla de entrenamiento": origen.data.params["datos_tabla"],
    "   snapshot de la OBT usado": origen.data.params["datos_obt_construida"],
    "   fecha de corte / filas / hash": (f"{origen.data.params['datos_fecha_corte']} / {origen.data.params['datos_filas']}"
                                         f" / {origen.data.params['datos_hash']}"),
    "   OBT actual (construida, filas)": f"{obt['construida']} / {obt['filas']}",
    "5. tablas Gold de origen": "GOLD.FCT_SALES_DAILY, GOLD.DIM_STORE, GOLD.DIM_DATE, GOLD.DIM_HOLIDAY (spark/jobs/build_obt.py)",
}).to_frame("traza")

,traza
1. id_prediccion,d83fc46e-c577-473c-acfb-96760b30e4e7
tienda / fecha / lote,1 / 2015-06-20 / 1
pronostico / decision,4490 / mantener_plan
2. version del modelo,1
run_id,0e34d952530740ec900db92ec5831a15
3. corrida en MLflow,final_hgb_champion
corrida de entrenamiento,tuning_hgb_mejor (d6e81855a4cc42c9b9219b99d7e44a41)
4. tabla de entrenamiento,PSET2_ROSSMANN.OBT.OBT_SALES_DAILY
snapshot de la OBT usado,2026-11-10 23:42:19.751020
fecha de corte / filas / hash,2015-06-19 / 757018 / 52c38c026e77454c


Si la fecha de construcción de la OBT actual coincide con el snapshot registrado, el modelo se puede reproducir con la OBT
actual; si no, el hash de los datos permite comprobar si los datos cambiaron (`notebooks/05b_reproduccion.ipynb`).